# Supplemental integrated walkthrough
Follow the lifecycle of an evidence brief from retrieval to review. The exercises use the real Deep Agents harness, a small fictional corpus, and actual model responses. Run top to bottom using configured Ollama and PostgreSQL/OpenSearch services.

In [1]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory

from deep_agent_lab.labs import (
    async_lab,
    context_lab,
    delegation_lab,
    interpreter_lab,
    mcp_lab,
    memory_lab,
    shell_lab,
    summarization_lab,
    workspace_lab,
)
from deep_agent_lab.service import (
    AgentService,
    Decision,
    ReviewRequest,
    RunRequest,
    open_checkpointer,
)
from deep_agent_lab.settings import Settings

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path/"pyproject.toml").exists())
os.chdir(ROOT)
scratch = TemporaryDirectory()
settings = Settings(
     deep_lab_mode="live",
    deep_lab_state_dir=Path(scratch.name), langfuse_enabled=False,
)

## 1. Tools, MCP, and threads
The model issues a search call, a tool returns passages, and the answer cites one or more exact evidence tokens. Each thread holds its own history.

In [2]:
with open_checkpointer(settings) as checkpoints:
    service = AgentService(settings, checkpoints)
    try:
        answer = service.run(RunRequest(
            prompt="Who approves a production restore?", thread_id="lesson-1",
        ))
    finally:
        service.close()

assert "database operations lead" in answer.answer.casefold()
assert any(row.citation in answer.answer for row in answer.evidence)
assert not answer.unknown_citations
print(answer.answer)
print(answer.steps)

Based on the search results, **the database operations lead** approves a production restore.

From the backup policy excerpt: *"The database operations lead must approve a production restore."* [[fbac2beaa4c7]]

The process also requires:
1. Opening a restore request with the database operations team
2. Restoring into an isolated database first
3. Verifying the requested records
4. Agreeing on the recovery plan before changing production
['search_knowledge']


In [3]:
mcp = await mcp_lab()
assert "search" in mcp["tools"] and mcp["results"]
print(mcp)


{'tools': ['search'], 'results': ['{\n  "id": "eb721ead272932f268eda808804085cce511a3605719cc51cd3ccdb0f1d2ff5b",\n  "title": "backup-policy",\n  "source": "file:///C:/Users/imrah/Documents/deepagents/data/sample/backup-policy.md",\n  "position": 0,\n  "excerpt": "# Northstar database backup policy\\n\\nThis is fictional sample knowledge for the RAG Lab.\\n\\nProduction PostgreSQL databases receive a full backup every day at 02:00 UTC. Transaction logs are archived continuously to support point-in-time recovery. Daily backups are retained for 30 days and monthly backups are retained for 12 months.\\n\\nTo restore accidentally deleted customer records, open a restore request with the database operations team. The database operations lead must approve a production restore. Restore into an isolated database first, verify the requested records, and agree on the recovery plan before changing production.\\n\\nThe recovery point objective is 15 minutes. The recovery time objective is four hou

## 2. Execution environment
Route reference paths to a temporary directory and draft paths to graph state. Compare a single fixed shell example with a QuickJS calculation.

In [4]:
workspace = workspace_lab()
assert "alternate.md" not in workspace["host_files"]
print(workspace)
print(shell_lab())
print(interpreter_lab())


{'results': ['1  Reference policies require reviewed changes.', 'Error: permission denied for write on /reference/alternate.md', 'Updated file /draft.md'], 'host_files': ['policy.md'], 'state_files': ['/draft.md']}
{'command': 'type policy.txt', 'exit_code': 0, 'output': 'Restore approval recorded.'}


C:\Users\imrah\Documents\deepagents\src\deep_agent_lab\labs.py:98: LangChainBetaWarning: The class `CodeInterpreterMiddleware` is in beta. It is actively being worked on, so the API may change.
  model=course_model(), middleware=[CodeInterpreterMiddleware(subagents=False, timeout=2)]


{'results': ['<stdout>\n90\n</stdout>\n<result>null</result>'], 'answer': 'The sum of the policy review durations (30 + 15 + 45 minutes) is **90 minutes**.'}


## 3. Context, skills, and memory
A long tool result gets a path and preview. Compaction creates a separate history file. Shared project memory and thread state have different lifetimes.

In [5]:
offload = context_lab()
summary = summarization_lab()
assert offload["files"][0].startswith("/large_tool_results/")
assert summary["files"] == ["/conversation_history/summary-lab.md"]
print(offload["files"], summary["files"])

memory = memory_lab()
assert memory["A"]["files"] == ["/private.md"]
assert memory["B"]["files"] == []
print(memory["A"]["tool_results"][0])

['/large_tool_results/e5365aa6-de8b-41ae-b624-1764fb57dc78'] ['/conversation_history/summary-lab.md']


1  # Format
2  Use short bullets.


## 4. Plan and delegate
QuickJS dispatches two specialists in parallel. The main graph receives two short reports and updates its plan. Try adding a third topic in the lab source.

In [6]:
delegation = await delegation_lab()
observed = "\n".join(delegation["results"]+[delegation["answer"]]).casefold()
assert "incident commander" in observed and "database operations lead" in observed
assert "task" in delegation["steps"]
print(delegation)


{'results': ['Based on the search results, I was able to find relevant policy documents through the search_policy function. Let me compile the evidence from those search results:\n\n## Evidence on Restore Approval Processes and SEV-1 Coordination Protocols\n\n### 1. Restore Approval Processes\n\n**Valid Restore Request:**\n- **Evidence ID**: `[[fbac2beaa4c7]]`\n- **Source**: backup-policy.md\n- **Quote**: "To restore accidentally deleted customer records, open a restore request with the database operations team."\n- **What constitutes a valid request**: A formal restore request must be opened through the appropriate channel (database operations team for production databases)\n\n**Who Approves Restores:**\n- **Evidence ID**: `[[fbac2beaa4c7]]`\n- **Source**: backup-policy.md\n- **Quote**: "The database operations lead must approve a production restore."\n- **Approval authority**: The database operations lead has final approval authority for production restores\n\n**Typical SLAs:**\n- **

## 5. Capstone
Draft a brief, observe the skill read and approval request, then close the checkpoint connection. Reopen the store before approving. This simulates saving after a CLI restart.

In [7]:
with open_checkpointer(settings) as checkpoints:
    service = AgentService(settings, checkpoints)
    try:
        proposal = service.run(RunRequest(
            prompt="Draft and save an evidence brief about production restore approval.",
            thread_id="capstone",
        ))
    finally:
        service.close()

assert proposal.status == "interrupted"
assert "read_file" in proposal.steps
print(proposal.reviews[0].action_requests[0]["args"]["markdown"])

# Evidence Brief: Production Restore Approval

## Question
What are the requirements and procedures for approving a production database restore in Northstar?

## Findings

### Approval Authority
The database operations lead must approve any production restore request [[fbac2beaa4c7]].

### Pre-restore Requirements
Before changing production, teams must:
1. Open a restore request with the database operations team
2. Restore into an isolated database first
3. Verify the requested records
4. Agree on the recovery plan [[fbac2beaa4c7]]

### Backup Retention and Recovery Targets
- Production PostgreSQL databases receive full backups daily at 02:00 UTC
- Transaction logs are archived continuously for point-in-time recovery
- Daily backups retained for 30 days; monthly backups retained for 12 months [[fbac2beaa4c7]]
- Recovery Point Objective (RPO): 15 minutes
- Recovery Time Objective (RTO): four hours [[fbac2beaa4c7]]

### Testing Requirements
Test a full restore once each quarter and docum

In [8]:
with open_checkpointer(settings) as checkpoints:
    service = AgentService(settings, checkpoints)
    try:
        result = service.resume("capstone", ReviewRequest(
            interrupt_id=proposal.reviews[0].id,
            decisions=[Decision(type="approve")],
        ))
        brief_id = result.saved_brief_ids[0]
        saved = service.store.get(brief_id)
        assert saved and saved.thread_id == "capstone"
        print(saved.title, saved.id)
    finally:
        service.close()

print(async_lab())


Production Restore Approval Evidence Brief 2acfe6181fcccd5a057ea0468d82a697


{'worker_running': True, 'worker_answer': 'An evidence gap refers to a situation where relevant research exists but has not yet been conducted or published to address a specific question, whereas an unavailable retrieval occurs when existing studies are known to exist but cannot be located due to access restrictions, search limitations, or loss of records.'}


## 6. Managed deployment
Open [the managed project walkthrough](../docs/managed-deployment.md). Sketch how the managed runtime would reach your search service and which store would own approved briefs. The deployment exercise is a design review.

### Optional live run
Initialize and ingest the course corpus, check `deep-lab doctor`, then set `RUN_LIVE` to `True`. This query reads evidence from your existing corpus.

In [9]:
RUN_LIVE = True
if RUN_LIVE:
    root = Path.cwd()
    if root.name == "notebooks":
        root = root.parent
    live = Settings(_env_file=root / ".env", deep_lab_mode="live", langfuse_enabled=False)
    live.deep_lab_state_dir = Path(scratch.name) / "extension"
    if not live.deep_lab_state_dir.is_absolute():
        live.deep_lab_state_dir = root / live.deep_lab_state_dir
    with open_checkpointer(live) as checkpoints:
        service = AgentService(live, checkpoints)
        try:
            result = service.run(RunRequest(prompt="Who approves a production restore?"))
            print(result.model_dump_json(indent=2))
        finally:
            service.close()

scratch.cleanup()


{
  "thread_id": "5a824f0d641b469b8c35a1974b12ccdf",
  "status": "completed",
  "answer": "Based on the search results, **the database operations lead** approves a production restore.\n\nFrom the backup policy excerpt: *\"The database operations lead must approve a production restore.\"* [[fbac2beaa4c7]]\n\nThe process also requires:\n1. Opening a restore request with the database operations team\n2. Restoring into an isolated database first\n3. Verifying the requested records\n4. Agreeing on the recovery plan before changing production",
  "evidence": [
    {
      "id": "8689ba038dbf7103d7964482372c286fecd73e9c2441db434affdc80f4d751a9",
      "title": "access-policy",
      "source": "file:///C:/Users/imrah/Documents/deepagents/data/sample/access-policy.md",
      "position": 0,
      "excerpt": "# Northstar access management\n\nThis is fictional sample knowledge for the RAG Lab.\n\nNew employees request application access through the internal service desk. Their manager approves the